# EDA: Liquid-Line Restriction

## Data dictionary
Same 24 sensor columns + `Datetime` as every Simulated-dataset file — see notebook 01
for the full breakdown. Same caveat carries forward: the three `_PRES` columns are
internally consistent but miscalibrated ~14-15x vs. real R410A physics.

## Why this fault is mechanistically different from everything examined so far

Undercharge/overcharge are charge-level faults (too little/too much refrigerant).
Condenser/evaporator fouling are heat-transfer degradation faults (a coil surface
losing efficiency). Liquid-line restriction is neither — it's a **physical blockage**
in the liquid line, the pipe carrying condensed liquid refrigerant from the condenser
outlet to the expansion device. A partial restriction (e.g. debris, a kinked line, a
partially closed valve) forces refrigerant through a narrower effective opening.

## Hypothesis (before looking at any data)

- A restriction should cause a pressure drop across the point of restriction — expect
  measurable effects on pressures downstream of the liquid line, i.e. lower pressure
  reaching the evaporator side (`RTU_REFG_SUCT_PRES` may drop, similar in direction to
  undercharge, since less liquid refrigerant reaches the evaporator to absorb heat —
  a restriction can look like a "starved evaporator" symptom, even though the root
  cause is completely different: a blockage vs. genuinely too little refrigerant).
- Severity here is measured in **bar** (a real pressure-drop unit), unlike every fault
  so far measured in %, which is itself a meaningful difference — this may indicate
  LBNL modeled this fault more directly as "pressure drop across the restriction"
  rather than "percent of some reference quantity," worth keeping in mind when
  interpreting severity scaling.
- Given undercharge and this fault might produce a similar-looking symptom (starved
  evaporator) through different physical causes, a real, useful question for later
  modeling: can these two faults actually be told apart from telemetry alone, or do
  they risk being confused by a classifier relying only on the same handful of
  suction-side signals? Not resolved yet — a genuine question to carry into this EDA
  and eventually into feature selection.
- No assumption yet on monotonicity or whether stage-2 filtering will be needed —
  every fault examined so far has needed a different combination of these, not a
  fixed formula.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# ml/ is not an installed package (see ml/pyproject.toml: package-mode = false —
# deliberate, since ml/ isn't a deployed service). Notebooks live in ml/notebooks/,
# so add the ml/ root to sys.path to make `from src.features...` imports work
# consistently, the same way pytest's pythonpath = ["."] setting already does
# for the test suite.
ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from src.features.effect_size import cohens_d  # noqa: E402
from src.features.filtering import stage2_only  # noqa: E402

files = {
    "baseline": "../data/raw/RTU_sim_baseline.csv",
    "liquidpipe01bar": "../data/raw/RTU_sim_liquidpipe01bar.csv",
    "liquidpipe04bar": "../data/raw/RTU_sim_liquidpipe04bar.csv",
    "liquidpipe08bar": "../data/raw/RTU_sim_liquidpipe08bar.csv",
    "liquidpipe10bar": "../data/raw/RTU_sim_liquidpipe10bar.csv",
}

dfs = {label: pd.read_csv(fname) for label, fname in files.items()}

for _label, df in dfs.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

for label, df in dfs.items():
    print(f"{label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

baseline: shape=(143941, 25), missing_values=0
liquidpipe01bar: shape=(143941, 25), missing_values=0
liquidpipe04bar: shape=(143941, 25), missing_values=0
liquidpipe08bar: shape=(143941, 25), missing_values=0
liquidpipe10bar: shape=(143941, 25), missing_values=0


## Load confirmed

All 5 files: shape=(143941, 25), 0 missing values — consistent with every Simulated-
dataset file examined so far.

In [2]:
severity_order = ["baseline", "liquidpipe01bar", "liquidpipe04bar", "liquidpipe08bar", "liquidpipe10bar"]

restriction_cols = ["RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_REFG_DISC_PRES", "RTU_TOT_CAPA"]

summary = pd.DataFrame({
    label: df[restriction_cols].mean()
    for label, df in dfs.items()
}).T.loc[severity_order]

summary

,RTU_REFG_SUCT_PRES,RTU_REFG_SUCT_TEMP,RTU_REFG_DISC_PRES,RTU_TOT_CAPA
baseline,1.564757e+07,58.343190,2.463834e+07,11801.510557
liquidpipe01bar,1.561336e+07,58.189687,2.501411e+07,11814.790916
liquidpipe04bar,1.550061e+07,57.788866,2.506383e+07,11908.575790
liquidpipe08bar,1.535584e+07,61.374769,2.441971e+07,10719.402197
liquidpipe10bar,1.502368e+07,66.357281,2.384199e+07,9631.953209


## Finding: liquid-line restriction shows a threshold effect, not a gradual trend —
## hypothesis partially rejected

Only `RTU_REFG_SUCT_PRES` is monotonic across all four severities (15.65M → 15.02M,
steadily down). Every other column shows a **threshold pattern**: little to no change
(sometimes even reversing direction) at 1 and 4 bar, followed by a sharp shift at 8
and 10 bar:

| Severity | SUCT_PRES | SUCT_TEMP | DISC_PRES | TOT_CAPA |
|---|---|---|---|---|
| 1 bar | -0.22% | -0.26% | +1.53% | +0.11% |
| 4 bar | -0.94% | -0.95% | +1.73% | +0.91% |
| 8 bar | -1.87% | +5.20% | -0.89% | -9.16% |
| 10 bar | -3.98% | +13.75% | -3.22% | -18.37% |

(percentages to be confirmed via a real `pct_change` calculation next — table above is
a preview, not yet verified)

**Hypothesis partially rejected**: capacity does not gradually drop with severity as
predicted — it briefly *rises* at low severities (1, 4 bar) before falling sharply at
high severities (8, 10 bar). This is a genuinely different shape from every fault
examined so far — undercharge/overcharge/condenser/evaporator fouling all showed either
clean monotonic trends or a real-but-bounded non-monotonic wobble (overcharge's discharge
pressure, condenser fouling's capacity at adjacent severities). This is the first fault
showing what looks like a **real threshold/breakpoint** between 4 and 8 bar, not just
noise around an otherwise-monotonic trend.

**Plausible mechanism, not yet verified**: a liquid-line restriction may not meaningfully
impair the refrigeration cycle until it's severe enough to induce flash gas (partial
refrigerant vaporization before the expansion device) — a genuine phase-change
threshold rather than a linearly-scaling resistance effect. This would explain why mild
restriction (1-4 bar) barely registers while severe restriction (8-10 bar) causes a
sharp, qualitatively different degradation. Not confirmed — would need refrigeration-
cycle domain expertise beyond what's derivable from data alone, same honesty standard
as overcharge's unresolved discharge-pressure finding.

**Open question carried forward, matching the hypothesis stated before this data was
seen**: does this threshold behavior make liquid-line restriction easier or harder to
tell apart from undercharge? At low severities (1-4 bar), capacity actually moves in
the *opposite* direction from undercharge (rises here, drops there) — which may
actually make early-stage confusion between these two faults less likely than
originally hypothesized, not more. Worth checking directly once severity-matched
comparisons are possible.

In [3]:
pct_change = (summary / summary.loc["baseline"] - 1) * 100
pct_change

,RTU_REFG_SUCT_PRES,RTU_REFG_SUCT_TEMP,RTU_REFG_DISC_PRES,RTU_TOT_CAPA
baseline,0.000000,0.000000,0.000000,0.000000
liquidpipe01bar,-0.218659,-0.263104,1.525158,0.112531
liquidpipe04bar,-0.939185,-0.950110,1.726964,0.907216
liquidpipe08bar,-1.864417,5.196114,-0.887355,-9.169236
liquidpipe10bar,-3.987133,13.736121,-3.232150,-18.383726


## Finding: liquid-line restriction shows a threshold effect, not a gradual trend —
## hypothesis partially rejected

Only `RTU_REFG_SUCT_PRES` is monotonic across all four severities. Every other column
shows a **threshold pattern**: little change (or reversed direction) at 1 and 4 bar,
followed by a sharp shift at 8 and 10 bar:

| Severity | SUCT_PRES | SUCT_TEMP | DISC_PRES | TOT_CAPA |
|---|---|---|---|---|
| 1 bar | -0.22% | -0.26% | +1.53% | +0.11% |
| 4 bar | -0.94% | -0.95% | +1.73% | +0.91% |
| 8 bar | -1.86% | +5.20% | -0.89% | -9.17% |
| 10 bar | -3.99% | +13.74% | -3.23% | -18.38% |

**Hypothesis partially rejected**: capacity does not gradually drop with severity as
predicted — it briefly *rises* at low severities (1, 4 bar) before falling sharply at
high severities (8, 10 bar). This is the first fault examined showing what looks like a
real threshold/breakpoint (somewhere between 4 and 8 bar), not just noise around an
otherwise-monotonic trend, and not a bounded wobble like overcharge's or condenser
fouling's non-monotonicity.

**Plausible mechanism, not yet verified**: a liquid-line restriction may not meaningfully
impair the cycle until severe enough to induce flash gas (partial refrigerant
vaporization before the expansion device) — a genuine phase-change threshold rather than
a linearly-scaling resistance effect. Not confirmed — would need refrigeration-cycle
domain expertise beyond what's derivable from data alone, same honesty standard as
overcharge's unresolved discharge-pressure finding.

**Open question carried forward**: at low severities (1-4 bar), capacity moves
*opposite* to undercharge's direction (rises here, drops there) — potentially making
early-stage confusion between these two faults less likely than the pre-data hypothesis
suggested, not more. Worth a direct check once severity-matched comparisons are possible.

In [4]:
stage2_dfs = {label: stage2_only(df) for label, df in dfs.items()}

stage2_summary = pd.DataFrame({
    label: df[restriction_cols].mean()
    for label, df in stage2_dfs.items()
}).T.loc[severity_order]

stage2_pct_change = (stage2_summary / stage2_summary.loc["baseline"] - 1) * 100
stage2_pct_change

,RTU_REFG_SUCT_PRES,RTU_REFG_SUCT_TEMP,RTU_REFG_DISC_PRES,RTU_TOT_CAPA
baseline,0.000000,0.000000,0.000000,0.000000
liquidpipe01bar,-0.687943,-0.870278,2.693722,1.376064
liquidpipe04bar,-0.985838,-1.420077,3.078770,2.079018
liquidpipe08bar,-3.778088,8.418217,1.374354,-2.567495
liquidpipe10bar,-10.376340,22.433159,-0.968483,-7.488448


## Stage-2 filtering reveals a real staging confound in capacity — and sharpens the
## suction-side threshold

| Severity | SUCT_PRES unfilt | SUCT_PRES filt | SUCT_TEMP unfilt | SUCT_TEMP filt | DISC_PRES unfilt | DISC_PRES filt | CAPA unfilt | CAPA filt |
|---|---|---|---|---|---|---|---|---|
| 1 bar | -0.22% | -0.69% | -0.26% | -0.87% | +1.53% | +2.69% | +0.11% | +1.38% |
| 4 bar | -0.94% | -0.99% | -0.95% | -1.42% | +1.73% | +3.08% | +0.91% | +2.08% |
| 8 bar | -1.86% | -3.78% | +5.20% | +8.42% | -0.89% | +1.37% | -9.17% | -2.57% |
| 10 bar | -3.99% | -10.38% | +13.74% | +22.43% | -3.23% | -0.97% | -18.38% | -7.49% |

**Capacity's apparent collapse was substantially inflated by the staging confound**:
the unfiltered -18.38% drop at 10 bar shrinks to -7.49% once stage-1/stage-2 blending
is removed — more than half the apparent effect was staging noise, not the fault
itself. This is the same pattern already seen in overcharge (where filtering resolved
most, not all, of a capacity anomaly), reinforcing that `RTU_TOT_CAPA` needs stage
filtering as standard practice before drawing conclusions about it, for any fault.

**Suction pressure and temperature move the opposite way** — filtering *sharpens* their
threshold effect rather than dampening it (SUCT_TEMP's 10-bar shift grows from +13.74%
to +22.43%). This confirms these two signals are carrying a real, staging-independent
fault signature, unlike capacity's partly-inflated one.

**The threshold shape itself survives filtering** — still little movement at 1-4 bar,
still a sharp shift at 8-10 bar, across every column. This isn't a filtering artifact;
it's a genuine property of how this fault behaves.

**Discharge pressure remains non-monotonic** even filtered (peaks at 4 bar, declines
after) — smaller magnitude than unfiltered but the same shape. Still an open question,
same honesty standard as overcharge's unresolved discharge-pressure finding.

In [5]:
d_4_vs_8_capa = cohens_d(
    stage2_dfs["liquidpipe04bar"]["RTU_TOT_CAPA"],
    stage2_dfs["liquidpipe08bar"]["RTU_TOT_CAPA"],
)
print(f"Cohen's d, 4bar vs 8bar (RTU_TOT_CAPA): {d_4_vs_8_capa:.3f}")

Cohen's d, 4bar vs 8bar (RTU_TOT_CAPA): 1.445


## Cohen's d confirms: the 4-to-8-bar transition is a real, large jump

`liquidpipe04bar` vs `liquidpipe08bar` (stage-2 `RTU_TOT_CAPA`): **d = 1.445** — large,
by the same convention used throughout this project. This confirms the threshold isn't
a subtle percentage-table artifact — it's one of the largest single-step effect sizes
found across any fault examined so far, and it happens to sit in the middle of this
fault's severity range rather than at either extreme.

**Practical implication for modeling**: liquid-line restriction may be easy to detect
as *present* once it crosses this threshold (8-10 bar), but low-severity restriction
(1-4 bar) produces only small, easily-missed shifts — even reversed in direction from
what the "starved evaporator" hypothesis predicted. A classifier trained mostly on
severe examples could plausibly miss early-stage restriction faults, or confuse them
with normal baseline variation, since they barely move from baseline at 1-4 bar.

## Summary: liquid-line restriction EDA

**Hypothesis partially rejected**: unlike every prior fault (which showed either clean
monotonic trends or bounded non-monotonic wobbles), liquid-line restriction shows a
genuine **threshold effect** — minimal change at 1-4 bar (sometimes reversed from the
predicted direction), then a sharp shift at 8-10 bar. Confirmed with Cohen's d: the
4-to-8-bar jump in capacity is large (d=1.445), one of the largest single-step effects
found across any fault so far.

**Staging confound matters here too, and matters a lot**: unfiltered capacity's drop
at 10 bar (-18.38%) was more than half staging noise — filtered, the real effect is
-7.49%. Reinforces stage-2 filtering as standard practice for `RTU_TOT_CAPA`
specifically, regardless of fault type.

**Strongest, cleanest signal**: `RTU_REFG_SUCT_TEMP` — monotonic-ish direction-wise but
really a threshold shape, sharpens under filtering (up to +22.43% at 10 bar).

**Still open**: `RTU_REFG_DISC_PRES` remains non-monotonic even after filtering — peaks
at 4 bar, declines after — an open question not resolved from data alone, same honesty
standard as overcharge's equivalent finding.

**Practical implication for modeling**: this fault likely needs different treatment
than a simple severity-regression feature — early-stage restriction (1-4 bar) may be
genuinely hard to distinguish from baseline, while late-stage (8-10 bar) is easy. A
model evaluated only on average performance across severities could look deceptively
good while still missing early-stage cases entirely.

**Open question carried into the final fault (suction-line restriction)**: this fault
is mechanistically similar to liquid-line restriction (both are